# Jobs 50053882 and 50008605 — the last rung of the N=96 ladder, and what warmup repaired

Two one-change runs that close out the leaked-order (N, ff) grid at N=96.

**Headline 1 (finding 31, `50053882`).** `L=7/ff=1024` — the only cell that could narrow the
knee from (6, 8] — clears the 0.90 depth gate in **5 of 6** rho=0.60 bins, so **L\*(96, ff=1024)
= 8 stands**. But on the n=1500 shared pool L=7 scores **0.9120** against L=8's **0.9113**:
**+0.0007, 92–91 discordant, p = 1.0**, a flat tie at 5.79M parameters against 6.58M. The gate
and the pool disagree about the same pair of cells, and the gate's verdict rests on **three
thousandths** in one bin (T=8, 0.897).

**Headline 2 (finding 32, `50008605`).** Five epochs of LR warmup move the deep-thin cell
`L=12/ff=256` from **0.8100 to 0.8900** on the same pool (**+0.0800, 203–83, p = 9.1e-13**), and
its n-weighted well-formed rate at rho=0.60 from **0.868 to 0.924**. The control held: the
shallow-thin cell `L=8/ff=256`, which never had a grammar problem, **ties its cold self**
(−0.0040, p = 0.765). So a large part of finding 25's deep-thin deficit was **trainability, not
capacity** — and yet the warm cell still clears the gate in only **3 of 6** bins, so
**`L*(96, ff=256) > 12` stands** and the MLP-width cost stays super-multiplicative in N.

**The unpre-registered consequence.** Inside the thin column the depth ordering **reverses**:
cold, L=12 was 3.1 points *below* L=8/ff=256; warm, it is **+0.0493 above it** (176–102,
p = 1.1e-05). Finding 25's "adding four more layers to the thin model makes it worse still" was
an artefact of how the deep cell was trained, not a statement about depth.

Both runs are `shuffle_adj=False`, so **every solve rate here is a decoding score**, not a
deduction score (finding 16). They extend the leaked-order ladder; the leak-free ladder is a
separate series (`50099945`).

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

S = os.environ["SCRATCH"] + "/spin_gnn"
L07 = f"{S}/sudoku_n96_L07_2026-10-28"        # 50053882, finding 31
WARM = f"{S}/sudoku_n96_warmup_2026-10-28"    # 50008605, finding 32
WIDE = f"{S}/sudoku_n96_layer_scan_2026-10-26"  # 49639312, finding 22
COLD = f"{S}/sudoku_n96_ff256_2026-10-27"     # 49816217, finding 25
pd.set_option("display.width", 220)

REF = "d0256_h04_L08_ff01024_lr1e-03_pre"     # the wide knee, in every pool as the reference

# Short labels for the eight N=96 cells this notebook touches.
LAB = {"d0256_h04_L04_ff01024_lr1e-03_pre": "L=4 ff=1024",
       "d0256_h04_L06_ff01024_lr1e-03_pre": "L=6 ff=1024",
       "d0256_h04_L07_ff01024_lr1e-03_pre": "L=7 ff=1024 (NEW)",
       "d0256_h04_L08_ff01024_lr1e-03_pre": "L=8 ff=1024 (knee)",
       "d0256_h04_L12_ff01024_lr1e-03_pre": "L=12 ff=1024",
       "d0256_h04_L08_ff00256_lr1e-03_pre": "L=8 ff=256 cold",
       "d0256_h04_L08_ff00256_lr1e-03_pre_w05": "L=8 ff=256 WARM (NEW)",
       "d0256_h04_L12_ff00256_lr1e-03_pre": "L=12 ff=256 cold",
       "d0256_h04_L12_ff00256_lr1e-03_pre_w05": "L=12 ff=256 WARM (NEW)"}


def solved(d):
    # Single quotes: this cell is a RAW triple-double-quoted string in the
    # builder, so a \" inside it keeps its backslash and lands in the notebook.
    '''Per-instance correctness on the shared pool, keyed by checkpoint tag.'''
    z = np.load(f"{d}/sharedpool_n1500_per_instance.npz", allow_pickle=True)
    return dict(zip([str(x) for x in z["names"]], z["solved"]))


def mcnemar(a, b):
    '''Exact two-sided McNemar on paired booleans -> (a-only, b-only, p).'''
    import math
    n10, n01 = int((a & ~b).sum()), int((b & ~a).sum())
    m, k = n10 + n01, min(n10, n01)
    p = min(1.0, 2 * sum(math.comb(m, i) for i in range(k + 1)) / 2 ** m) if m else 1.0
    return n10, n01, p

## 1. Metric of record: the shared n=1500 pool

`sudoku_agg_eval` builds one pool at `seed_start=50000000000`, far above every training seed, and
instance *i* is drawn at `seed_start + i`, so a larger n is a strict superset of a smaller one.
Every N=96 cell in this notebook has `shuffle_adj=False`, `hint_mode=removal`, degree 5 and q=4,
so the prompts are byte-identical across cells and **every comparison below is paired instance by
instance**. The wide knee `L=8/ff=1024` appears in both runs' pools as the reference.

*Free replication check:* the reference checkpoint's 1500-instance correctness vector is
**identical** in the 10-28 L=7 pool and the 10-28 warmup pool — two independent `agg_eval`
invocations, byte-for-byte the same answers, which is the third time this checkpoint has
reproduced (finding 25 recorded the first two).

In [ ]:
cols = ["checkpoint", "solve_rate", "wilson_lo", "wilson_hi", "delta",
        "discordant_won", "discordant_lost", "mcnemar_exact_p",
        "truncated_rate", "outcome_accuracy"]
for name, d in (("finding 31: L=7 run", L07), ("finding 32: warmup run", WARM),
                ("finding 22: the wide ladder it joins", WIDE)):
    t = pd.read_csv(f"{d}/sharedpool_n1500.csv")
    t["checkpoint"] = t.checkpoint.map(lambda c: LAB.get(c, c))
    print(f"=== {name}  (delta and p are vs that pool's own reference)")
    print(t[cols].round(4).to_string(index=False), "\n")

sv_l07, sv_warm = solved(L07), solved(WARM)
print("reference correctness vector identical across the two runs:",
      bool((sv_l07[REF] == sv_warm[REF]).all()))

In [ ]:
# The pairs the CSVs do not contain: each pool is scored against its own reference
# only, so adjacent-rung and warm-vs-cold contrasts have to be recomputed here.
PAIRS = [("L=7 vs L=6  (wide, adjacent rungs)",
          sv_l07["d0256_h04_L07_ff01024_lr1e-03_pre"],
          sv_l07["d0256_h04_L06_ff01024_lr1e-03_pre"]),
         ("L=7 vs L=8  (wide, the knee)",
          sv_l07["d0256_h04_L07_ff01024_lr1e-03_pre"], sv_l07[REF]),
         ("L=12/ff256 WARM vs its COLD self  (the treatment)",
          sv_warm["d0256_h04_L12_ff00256_lr1e-03_pre_w05"],
          sv_warm["d0256_h04_L12_ff00256_lr1e-03_pre"]),
         ("L=8/ff256 WARM vs its COLD self  (the CONTROL)",
          sv_warm["d0256_h04_L08_ff00256_lr1e-03_pre_w05"],
          sv_warm["d0256_h04_L08_ff00256_lr1e-03_pre"]),
         ("L=12/ff256 WARM vs L=8/ff256 cold  (the reversal)",
          sv_warm["d0256_h04_L12_ff00256_lr1e-03_pre_w05"],
          sv_warm["d0256_h04_L08_ff00256_lr1e-03_pre"]),
         ("L=12/ff256 WARM vs the wide knee",
          sv_warm["d0256_h04_L12_ff00256_lr1e-03_pre_w05"], sv_warm[REF])]
for lab, a, b in PAIRS:
    w, l, p = mcnemar(a, b)
    print(f"{lab:52s} delta={a.mean() - b.mean():+.4f}  {w:3d}-{l:3d}  p={p:.3g}")

### What that table says

| contrast | delta | discordant | p |
|---|---|---|---|
| L=7 vs L=6 (wide) | **+0.0287** | 126–83 | 0.0036 |
| L=7 vs L=8 (wide knee) | **+0.0007** | 92–91 | **1.0** |
| L=12/ff=256 warm vs cold | **+0.0800** | 203–83 | 9.1e-13 |
| L=8/ff=256 warm vs cold — *the control* | −0.0040 | 137–143 | **0.765** |
| L=12/ff=256 warm vs L=8/ff=256 cold | **+0.0493** | 176–102 | 1.1e-05 |
| L=12/ff=256 warm vs wide knee | −0.0213 | 86–118 | 0.030 |

**The pool cannot separate L=7 from L=8.** 183 instances are discordant, so this is not an
underpowered test: it is a tie measured on 183 disagreements. L=7 is resolvably above L=6
(+0.0287) and indistinguishable from L=8 at 88% of its parameters. A ladder whose rungs are
defined by a 0.90 gate on depth bins therefore has a rung the pool does not see — stated plainly
because it is the one weakness in this ladder that no further N=96 run will fix.

**The warmup control is the load-bearing part of finding 32.** If warmup had lifted the whole
thin column, finding 25's 7.1-point width gap would have been measured against an undertrained
competitor and every N=96 width number would need re-running. It did not: the cell with no
grammar problem moved by −0.004 (p = 0.77), while the cell with one moved by +0.080.

## 2. Where L\* is read off: six depth bins at fixed realized density

L\* is the smallest `num_layers` reaching **0.90** on depth-T instances at fixed *realized*
density. `sudoku_depth_eval` bins a pool by DD depth T; at rho=0.60 the realized density is
**0.5938 in every bin of every cell**, so the bins differ in reasoning depth and in nothing else.
Bin sizes are n = 86, 146, 143, 69, 68, 43 for T = 4…9 — the two deepest bins are thin, which is
where the L=7 verdict is decided.

In [ ]:
dep = pd.concat([pd.read_csv(f"{WIDE}/depth_by_layers_n96.csv"),
                 pd.read_csv(f"{COLD}/depth_by_layers_n96_ff256.csv"),
                 pd.read_csv(f"{L07}/depth_n96_L07.csv"),
                 pd.read_csv(f"{WARM}/depth_n96_ff256_warmup.csv")])
d60 = dep[dep.rho_requested == 0.60]
tab = d60.pivot_table(index="T", columns="checkpoint", values="solve_rate")
tab.columns = [LAB.get(c, c) for c in tab.columns]
print("rho=0.60, realized density", sorted(d60.realized_rho.round(4).unique()))
print(tab.round(3).to_string())
print("\nn per bin:", d60.groupby("T").n.max().to_dict())

print("\ngate (bins >= 0.90), trend over the ordered bins:")
for c in sorted(d60.checkpoint.unique()):
    g = d60[d60.checkpoint == c].sort_values("T")
    k = int((g.solve_rate >= 0.90).sum())
    r = stats.pearsonr(g["T"], g.solve_rate)[0]
    x = np.repeat(g["T"].values, g.n.values)          # Cochran-Armitage: one row
    y = np.concatenate([np.r_[np.ones(int(round(s * n))), np.zeros(n - int(round(s * n)))]
                        for s, n in zip(g.solve_rate, g.n)])  # per instance, ordered bins
    z = np.corrcoef(x, y)[0, 1] * np.sqrt(len(x))
    print(f"  {LAB.get(c, c):24s} {k}/{len(g)}   r(solve,T)={r:+.3f}   "
          f"CA z={z:+.2f}  p={2 * stats.norm.sf(abs(z)):.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.3), sharey=True)
SERIES = [("the wide ladder, ff=1024", axes[0],
           [("d0256_h04_L04_ff01024_lr1e-03_pre", "0.75"),
            ("d0256_h04_L06_ff01024_lr1e-03_pre", "tab:orange"),
            ("d0256_h04_L07_ff01024_lr1e-03_pre", "tab:blue"),
            ("d0256_h04_L08_ff01024_lr1e-03_pre", "tab:green")]),
          ("the thin column, ff=256: warm vs cold", axes[1],
           [("d0256_h04_L08_ff00256_lr1e-03_pre", "0.6"),
            ("d0256_h04_L08_ff00256_lr1e-03_pre_w05", "tab:green"),
            ("d0256_h04_L12_ff00256_lr1e-03_pre", "tab:red"),
            ("d0256_h04_L12_ff00256_lr1e-03_pre_w05", "tab:purple")])]
for title, ax, cells_ in SERIES:
    for tag, colour in cells_:
        g = d60[d60.checkpoint == tag].sort_values("T")
        if not len(g):
            continue
        # Jeffreys interval per bin, so the thin T=8/T=9 bins are not read as points.
        lo, hi = [], []
        for _, r in g.iterrows():
            k = r.solve_rate * r.n
            a, b = stats.beta.interval(0.95, k + 0.5, r.n - k + 0.5)
            lo.append(max(0.0, r.solve_rate - a))
            hi.append(max(0.0, b - r.solve_rate))
        ls = "--" if tag.endswith("_w05") else "-"
        ax.errorbar(g["T"], g.solve_rate, yerr=[lo, hi], marker="o", lw=2, ls=ls,
                    color=colour, capsize=3, label=LAB[tag])
    ax.axhline(0.90, color="k", ls=":", lw=1.5)
    ax.set_title(title)
    ax.set_xlabel("DD depth T of the instance")
    ax.legend(fontsize=8, loc="lower left")
axes[0].set_ylabel("solve_rate on that bin")
axes[0].text(4.05, 0.905, "the 0.90 gate", size=8)
fig.tight_layout()

### Reading the gate

| cell | bins ≥ 0.90 | the misses | CA z (trend) |
|---|---|---|---|
| L=4 ff=1024 | 0 / 6 | all | −0.80 (p 0.42) |
| L=6 ff=1024 | 3 / 6 | the three **deepest** | −1.92 (p 0.055) |
| **L=7 ff=1024** | **5 / 6** | **T=8 only, at 0.897** | −1.07 (p 0.28) |
| L=8 ff=1024 | 6 / 6 | — | −0.23 (p 0.81) |
| L=8 ff=256 cold | 3 / 6 | T=6, 7, 8 | −2.22 (p 0.026) |
| L=8 ff=256 **warm** | 3 / 6 | T=6, 8, and T=5 at 0.911→no | −1.52 (p 0.13) |
| L=12 ff=256 cold | 0 / 6 | all (0.829–0.868, flat) | +0.67 (p 0.50) |
| L=12 ff=256 **warm** | 3 / 6 | T=5, 7, 9 | −1.54 (p 0.12) |

**Finding 31's pre-registration (a) predicted 4–5 of 6 and got 5**, so L\*(96, ff=1024) = 8 —
by **0.003** in the T=8 bin (n=68, so one instance is 0.015). Three thousandths is not a
measurement; it is where a threshold happened to fall. The honest statement is that the N=96
width factor should be quoted as **">1.5x, denominator resting on a 0.003 bin margin"**, and
that the pool calls L=7 and L=8 a tie outright.

**Finding 31's (b) is weak, not confirmed:** the single miss *is* the second-deepest bin, which
has the predicted shape, but with one miss there is nothing to trend — CA z = −1.07, p = 0.28.

**Warmup changes the deep-thin cell's failure shape, not just its level.** Cold L=12/ff=256 was
*flat and faintly rising* in T (0.829–0.868, z = +0.67) — the signature of a model whose failures
are not about reasoning depth at all, which is what the grammar diagnosis predicted. Warm, it
falls with T (z = −1.54) like an under-depth reasoner, and the bins it misses are no longer
"all of them".

## 3. Mechanism: what warmup actually repaired

`well_formed_rate` is the fraction of generations that parse as a trace at all. Finding 25's
diagnosis was that the deep-thin cell's deficit is mostly a *grammar* failure — it stops writing
valid traces — and that an early-training instability is the likely cause. Warmup is the direct
test of that: a 5-epoch linear ramp to the same 1e-3, nothing else changed.

In [ ]:
rows = []
for c in sorted(d60.checkpoint.unique()):
    g = d60[d60.checkpoint == c]
    wf = (g.well_formed_rate * g.n).sum() / g.n.sum()   # n-weighted over the six bins
    rows.append({"cell": LAB.get(c, c),
                 "well_formed (n-weighted)": round(wf, 3),
                 "worst bin": round(g.well_formed_rate.min(), 3),
                 "best bin": round(g.well_formed_rate.max(), 3),
                 "solve/well_formed, min": round((g.solve_rate / g.well_formed_rate).min(), 3),
                 "solve/well_formed, max": round((g.solve_rate / g.well_formed_rate).max(), 3)})
print(pd.DataFrame(rows).to_string(index=False))

# The pool agrees, on its own two generative statistics.
for d, tags in ((WARM, ["d0256_h04_L12_ff00256_lr1e-03_pre",
                        "d0256_h04_L12_ff00256_lr1e-03_pre_w05"]),):
    t = pd.read_csv(f"{d}/sharedpool_n1500.csv").set_index("checkpoint")
    print("\npool, generative statistics:")
    print(t.loc[tags, ["solve_rate", "truncated_rate", "outcome_accuracy"]].round(4)
          .rename(index=LAB).to_string())

### The repair is real, and it is partial

n-weighted over the six rho=0.60 bins, `L=12/ff=256` goes **0.868 → 0.924** well formed, and on
the pool `truncated_rate` **0.0167 → 0.0100** with `outcome_accuracy` **0.8687 → 0.9300**. Three
independent generative statistics move the same way, and the control cell's barely move. So
finding 25's mechanism was right: a large part of the deep-thin deficit was the model failing to
*write* traces, not failing to deduce.

**But pre-registration (a) is falsified, and in a way worth naming.** It asked for
`well_formed_rate >= 0.98` in **every** bin. The n-weighted rate is 0.924, and the **worst single
bin goes 0.853 (cold) → 0.814 (warm)** — i.e. warmup raised the *average* well-formedness while
leaving the worst bin no better than before. An average that improves while the floor does not is
the signature of a partial repair, not a fixed cell, and the earlier one-line write-up of this
finding compared the cold *worst bin* (0.853) against the warm *weighted mean* (0.923), which is
not a like-for-like pair. Both numbers are in the table above.

**(b) is falsified too, which is the result that matters for the grid:** the warm cell clears the
0.90 gate in 3 of 6 bins, so `L*(96, ff=256) > 12` **stands**, and the MLP-width factor on L\*
stays `1.00x / 1.50x / >1.50x` at N = 24 / 48 / 96 — super-multiplicative in N, with one confound
now removed rather than suspected.

## 4. `val_nll` as a ranking statistic, checked rather than assumed

Finding 25 noted that `final_val_nll` ranked the three N=96 cells almost in reverse of their
pooled solve rate, and finding 32's pre-registration (d) asked whether warmup removes that
inversion. The earlier write-up left (d) unchecked; it is checkable from the two `sweep.csv`
files and is answered here.

In [ ]:
rows = []
for d in (WIDE, COLD, L07, WARM):
    sw = pd.read_csv(f"{d}/sweep.csv")
    pool_t = pd.read_csv(f"{d}/sharedpool_n1500.csv").set_index("checkpoint")
    for _, r in sw.iterrows():
        tag = os.path.basename(r.checkpoint_dir)
        rows.append({"cell": LAB.get(tag, tag), "L": int(r.num_layers),
                     "ff": int(r.dim_feedforward),
                     "params_M": round(r.n_params / 1e6, 2),
                     "warmup_epochs": int(r.get("warmup_epochs", 0)),
                     "final_val_nll": round(r.final_val_nll, 4),
                     "pool_solve_rate": round(float(pool_t.solve_rate.get(tag, np.nan)), 4)})
t = pd.DataFrame(rows).drop_duplicates("cell").sort_values("final_val_nll")
print(t.to_string(index=False))
print("\nSpearman(val_nll, pool solve_rate) over these cells:",
      round(stats.spearmanr(t.final_val_nll, t.pool_solve_rate).statistic, 3))

### (d): confirmed inside the thin column, falsified across columns

Cold, the thin column was inverted: `L=12/ff=256` had the **lower** `val_nll` (1.4108 vs 1.4337)
and the **lower** pool rate (0.8100 vs 0.8407). Warm, the inversion is gone *within that column* —
L=12 keeps the lower nll (1.4137 vs 1.4335) and now also has the higher pool rate (0.8900 vs
0.8367).

Across columns it survives and is as bad as ever: warm `L=12/ff=256` has a **lower** final
`val_nll` than the new `L=7/ff=1024` (1.4137 vs 1.4142) while scoring **2.2 points lower** on the
pool (0.8900 vs 0.9120). Teacher-forced next-token loss still does not rank architectures at
N=96, so finding 4's conclusion is unchanged: `val_nll` is not a model-selection signal here, and
the open "no best-val checkpoint selection" item cannot be resolved by it.

## 5. Pre-registrations, as resolved

Both sets were written into the sbatch headers before submission
(`scripts/slurm/sudoku2_n96_L07_wide.sbatch`, `scripts/slurm/sudoku2_n96_ff256_warmup.sbatch`).

### Finding 31 — `50053882`, L=7/ff=1024

| | claim | verdict | evidence |
|---|---|---|---|
| (a) | L=7 clears 4–5 of the six rho=0.60 bins, so L\*(96, ff=1024) = 8 stands | **CONFIRMED** | 5 of 6; the miss is T=8 at 0.897, three thousandths under the gate |
| (b) | the bins it misses are the deepest ones | **WEAK** | the single miss is the second-deepest bin, but one miss cannot trend: CA z = −1.07, p = 0.28, r(solve,T) = −0.339 |
| (c) | on the pool L=7 lands strictly between L=6 and L=8, resolvably below L=8 | **FALSIFIED** | +0.0007 vs L=8, 92–91, p = 1.0 — a tie, at 0.88x the parameters. Its own falsifier text: "the pool cannot separate adjacent rungs" |
| (d) | well_formed_rate ≥ 0.95 in every bin | **CONFIRMED** | 0.957–0.993, n-weighted 0.977. No wide-column grammar failure |

### Finding 32 — `50008605`, warmup on the thin column

| | claim | verdict | evidence |
|---|---|---|---|
| (a) | warm L=12/ff=256 reaches well_formed ≥ 0.98 in **every** rho=0.60 bin | **FALSIFIED** | n-weighted 0.924; worst bin 0.814, which is *below* the cold cell's 0.853 |
| (b) | given (a), it clears the 0.90 gate, so L\*(96, ff=256) ≤ 12 | **FALSIFIED** | 3 of 6 bins. `L*(96, ff=256) > 12` stands; the width factor stays super-multiplicative in N |
| (c) | the control L=8/ff=256 ties its cold self on the pool | **CONFIRMED** | −0.0040, 137–143, p = 0.765. Warmup did **not** lift the whole thin column, so finding 25's 7.1-point width gap was not measured against an undertrained competitor |
| (d) | warmup stops val_nll ranking the thin cells in reverse | **SPLIT** | fixed **inside** the thin column (L=12 now lower nll *and* higher pool rate); still inverted **across** columns (warm L=12 1.4137 beats L=7's 1.4142 on nll and loses by 0.022 on the pool) |

**Unpre-registered, and the reason this pair is worth a notebook:** warmup **reverses the depth
ordering inside the thin column**. Cold, L=12/ff=256 scored 3.1 points *below* L=8/ff=256 and
finding 25 read that as "depth cannot buy back MLP width at N=96". Warm, L=12 is **+0.0493 above**
L=8 (176–102, p = 1.1e-05). The direction of finding 25's conclusion survives — the thin column
still does not reach the wide knee, warm L=12 is −0.0213 against it (p = 0.030) — but its sharpest
sentence was about training, not architecture.

## 6. Provenance, and what these runs do not establish

| | finding 31 (`50053882`) | finding 32 (`50008605`) |
|---|---|---|
| commit | `6f8adc2` | `801c593` |
| launcher | `scripts/slurm/sudoku2_n96_L07_wide.sbatch` | `scripts/slurm/sudoku2_n96_ff256_warmup.sbatch` |
| cells trained | `L=7/ff=1024` | `L=12/ff=256 +w5`, `L=8/ff=256 +w5` |
| runtime | 06:49:55 | 15:52:12 |
| task | N=96, degree 5, q=4, `hint_mode=removal`, `shuffle_adj=False` | same |
| budget | 80000 instances × 150 epochs, batch 32, lr 1e-3 constant | same, plus a 5-epoch linear ramp |
| seed | 0 | 0 |
| output | `$SCRATCH/spin_gnn/sudoku_n96_L07_2026-10-28/` | `$SCRATCH/spin_gnn/sudoku_n96_warmup_2026-10-28/` |

**Caveats, in the order they would bite.**

1. **Leaked order.** `shuffle_adj=False`, so every number here is a decoding score: finding 16
   showed all three N=96 checkpoints collapse when `adj[v]` is permuted. This ladder measures how
   many layers the *leaked* task needs. The leak-free ladder is a separate series and is running
   at N=48 (`50099945`).
2. **One seed per cell.** Every entry in the (N, ff) grid is a single training run at seed 0. The
   warm-vs-cold contrast is two training runs, so "warmup did it" is one change, not a
   distribution over initialisations.
3. **The gate is sensitive at the third decimal.** L\*(96, ff=1024) = 8 rests on T=8 scoring 0.897
   in a 68-instance bin, while the pool calls L=7 and L=8 a tie on 183 discordant instances. The
   L\* definition and the metric of record disagree about this rung and the notebook does not
   resolve which to prefer.
4. **Warmup is not a tuned intervention.** Five epochs of linear ramp, chosen once, not scanned.
   "A longer ramp would clear the gate" is untested and nothing here rules it in or out.
5. **The deepest bins are thin** (n = 68 and 43), so per-bin rates there carry roughly ±0.07 at
   95%, which is why the trend tests are reported alongside the bin counts rather than instead of
   them.